# 十二个模型的统一比较

优先读取保存的逐行预测；首次缺少结果时自动补齐依赖模型。


In [ ]:
from experiment_pipeline import Pipeline
run = Pipeline()
globals().update(run.comparison())


# 8. 十二个模型的 Validation 与 Test 汇总

十二个模型的正式拟合均使用全部 Train，并在相同的 Validation / Test 行上评估。下方分别显示 Validation 表与 Test 表。只有树模型从固定的 20 万条 Train 样本选择超参数；本节只调用已有模型预测并计算指标，不重新训练。`RMSE` 和 `R²` 针对裁剪后的 `p_target` 计算；训练行数对齐后，模型特征、目标函数和结构仍有差异。

Train mean 是只预测 `p` 的常数基线，不提供半衰期预测，因此 `Spearman(h)` 留空。Loss 由各模型自己的训练目标定义，不适合跨模型直接比较；跨模型请看 MAE、RMSE、R² 等共同指标。


In [ ]:
from IPython.display import display
print("validation_summary")
display(validation_summary)
print("test_summary")
display(test_summary)


# 10. 不同复习间隔下的模型表现

## 10.1 分组与比较口径

使用相同的 Test 行，将 `delta_days` 划为 `[0,3)`、`[3,7)`、`[7,30)`、`[30,+∞)` 天。每组比较十二个模型的 `MAE(p)`、AUC 和 `Spearman(h)`。各组样本数不同；整体 MAE 等于按组内行数加权的 MAE，不能直接取四组的简单平均。


## 10.2 十二个模型按 delta 的 Test 对照

三张表分别比较 `MAE(p)`、AUC、`Spearman(h)`；十二个模型使用相同的 Test 行和全部 Train 进行正式拟合。折线图展示四个 delta 区间的趋势。E2/E4 的 `h_hat` 由预测回忆率和 `delta_days` 反推；神经模型的 `h_hat` 来自网络。

In [ ]:
import matplotlib.pyplot as plt

delta_model_order = model_order  # 第 8 节的十二个模型，顺序保持一致
labels = ["[0, 3) 天", "[3, 7) 天", "[7, 30) 天", "[30, +∞) 天"]
test_group_ids = np.searchsorted(np.array([3.0, 7.0, 30.0]),
                                 test_days, side="right")
test_delta_rows = []
for name in delta_model_order:
    p_pred, h_pred = test_predictions[name]
    for group_id, label in enumerate(labels):
        mask = test_group_ids == group_id
        test_delta_rows.append({
            "model": name, "delta 区间": label, "rows": int(mask.sum()),
            "MAE(p)": float(np.abs(p_pred[mask] - test_true_p[mask]).mean()),
            "AUC": auc_fixed(np.rint(test_true_p[mask]), p_pred[mask]),
            "Spearman(h)": (np.nan if name == "Train mean" else
                             spearman_with_ties(test_true_h[mask], h_pred[mask])),
        })
final_test_delta = pd.DataFrame(test_delta_rows)
for name in delta_model_order:
    part = final_test_delta[final_test_delta["model"] == name]
    overall = float(final_summary.loc[
        (final_summary["model"] == name) & (final_summary["split"] == "Test"),
        "MAE(p)",
    ].iloc[0])
    assert part["rows"].sum() == n_test
    assert np.isclose(np.average(part["MAE(p)"], weights=part["rows"]), overall)


delta_order = ["[0, 3) 天", "[3, 7) 天", "[7, 30) 天", "[30, +∞) 天"]
delta_counts = (final_test_delta[final_test_delta["model"] == "HLR"]
                .set_index("delta 区间").loc[delta_order, "rows"])
print("Test 各 delta 区间的样本数；十二个模型共用这些行。")
display(delta_counts.rename("rows").to_frame())

delta_matrices = {}
for metric in ("MAE(p)", "AUC", "Spearman(h)"):
    matrix = (final_test_delta.pivot(index="model", columns="delta 区间",
                                     values=metric)
              .loc[delta_model_order, delta_order])
    assert matrix.shape == (12, 4)
    if metric == "Spearman(h)":
        assert matrix.loc["Train mean"].isna().all()
        assert np.isfinite(matrix.drop(index="Train mean").to_numpy()).all()
    else:
        assert np.isfinite(matrix.to_numpy()).all()
    delta_matrices[metric] = matrix
    print(f"Test {metric}：逐行比较十二个模型；逐列比较同一复习间隔。")
    display(matrix.round(6))

plot_labels = ["0–<3d", "3–<7d", "7–<30d", "≥30d"]
colors = plt.get_cmap("tab20").colors
markers = ["o", "s", "^", "D", "v", "P", "X", "*", "h", "<", ">", "d"]
line_styles = ["-", "--", "-.", ":", "-", "--", "-.", ":", "-", "--", "-.", ":"]
x_centers = np.arange(4)
fig, axes = plt.subplots(1, 3, figsize=(18, 6.2), sharex=True)
for model_index, model_name in enumerate(delta_model_order):
    # 只横向错开绘图位置；每个点的 y 值仍是表中的真实指标。
    x_positions = x_centers + (model_index - 5.5) * 0.032
    for ax, metric in zip(axes, ("MAE(p)", "AUC", "Spearman(h)")):
        if model_name == "Train mean" and metric == "Spearman(h)":
            continue
        ax.plot(x_positions, delta_matrices[metric].loc[model_name].to_numpy(),
                marker=markers[model_index], linestyle=line_styles[model_index],
                linewidth=1.8, markersize=6, markeredgecolor="white",
                markeredgewidth=0.7, color=colors[model_index],
                label=model_name)
for ax, metric in zip(axes, ("MAE(p)", "AUC", "Spearman(h)")):
    ax.set_title(metric)
    ax.set_xticks(x_centers, plot_labels)
    ax.set_xlim(-0.25, 3.25)
    ax.set_xlabel("delta (days)")
    ax.grid(alpha=0.25)
axes[0].set_ylabel("lower is better")
axes[1].set_ylabel("higher is better")
axes[2].set_ylabel("higher is better")
handles, legend_labels = axes[0].get_legend_handles_labels()
fig.legend(handles, legend_labels, loc="lower center", ncol=4,
           bbox_to_anchor=(0.5, 0.02), frameon=False)
fig.suptitle('Twelve-model Test performance by review interval')
fig.subplots_adjust(bottom=0.34, wspace=0.28, top=0.86)
plt.show()
